# Gym Environment Example

This notebook demonstrates how to create and interact with a quantum circuit gymnasium environment where different actions are performed on circuits.

## Table of Contents
1. [Setup and Imports](#setup)
2. [Create a Dataset](#dataset)
3. [Create the Environment](#environment)
4. [Understanding Observation and Action Spaces](#spaces)
5. [Performing Different Actions](#actions)
6. [Complete Episode Example](#episode)
7. [Working with Validation Circuits](#validation)
8. [Custom Reward Functions](#reward)
9. [Multi-Qubit Environments](#multiqubit)
10. [Advanced: Action Strategies](#strategies)

## 1. Setup and Imports <a id="setup"></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rlnoise import (
    DatasetConfig,
    NoiseConfig,
    GymEnvConfig,
    RewardConfig,
    DatasetGenerator,
    CircuitEncoder,
    QuantumCircuitEnv,
    create_quantum_circuit_env,
)

# Set random seed for reproducibility
np.random.seed(42)

## 2. Create a Dataset <a id="dataset"></a>

First, we need a dataset of quantum circuits to work with.

In [ ]:
# Configure dataset
dataset_config = DatasetConfig(
    n_circuits=20,
    qubits=1,
    moments=10,
    clifford=True,
    seed=42
)

noise_config = NoiseConfig(
    primitive_gates=["rx", "rz"],
    dep_lambda=0.03,
    seed=42
)

# Generate dataset
generator = DatasetGenerator(dataset_config, noise_config)
dataset = generator.generate(verbose=True)

print(f"\nDataset shape: {dataset.circuits.shape}")
print(f"Circuit encoding dimension: {dataset.circuits.shape[2]}")

## 3. Create the Environment <a id="environment"></a>

Now let's create a gymnasium environment from our dataset.

In [ ]:
# Configure environment
env_config = GymEnvConfig(
    kernel_size=3,          # Sliding window size
    action_penalty=0.01,    # Penalty for taking actions
    val_split=0.2,          # 20% validation circuits
)

reward_config = RewardConfig(
    metric="trace",         # Use trace distance
    function="inverted",    # Transform: 1/(1+distance)
    alpha=20.0,            # Scaling factor
)

# Create environment
env = create_quantum_circuit_env(
    dataset=dataset,
    primitive_gates=["rx", "rz"],
    env_config=env_config,
    reward_config=reward_config,
)

print(f"Environment created!")
print(f"Training circuits: {env.n_circuits_train}")
print(f"Validation circuits: {env.n_validation_circuits}")

## 4. Understanding Observation and Action Spaces <a id="spaces"></a>

### Observation Space
The observation is a sliding window over the circuit encoding:
- Shape: `(encoding_dim, n_qubits, kernel_size)`
- Example: `(8, 1, 3)` means 8 features, 1 qubit, window of 3 gates

### Action Space
Actions represent noise parameters for each qubit:
- Shape: `(n_qubits, 4)`
- For each qubit: `[epsilon_x, epsilon_z, reset, depolarizing]`
- Values are in range [0, 1], scaled internally to [0, max_value]

In [ ]:
print("Observation Space:")
print(f"  Shape: {env.observation_space.shape}")
print(f"  Type: {env.observation_space.dtype}")
print(f"  Low: {env.observation_space.low[0, 0, 0]}")
print(f"  High: {env.observation_space.high[0, 0, 0]}")

print("\nAction Space:")
print(f"  Shape: {env.action_space.shape}")
print(f"  Type: {env.action_space.dtype}")
print(f"  Low: {env.action_space.low}")
print(f"  High: {env.action_space.high}")

# Sample observation and action
obs, info = env.reset()
action = env.action_space.sample()

print(f"\nSample observation shape: {obs.shape}")
print(f"Sample action shape: {action.shape}")
print(f"Sample action values: {action}")

## 5. Performing Different Actions <a id="actions"></a>

Let's demonstrate different types of actions on a single circuit.

### 5.1 Reset to Specific Circuit

In [ ]:
# Reset to circuit index 5
obs, info = env.reset(options={"circuit_idx": 5})

print(f"Reset to circuit {info['circuit_idx']}")
print(f"Circuit length: {info['circuit_length']} gates")
print(f"Initial position: 0")
print(f"\nInitial observation (first gate features):")
print(obs[:, 0, 0])  # Features for qubit 0, position 0

### 5.2 Random Actions

In [ ]:
# Take 3 random actions
print("Taking random actions:\n")

for i in range(3):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Step {i+1}:")
    print(f"  Action: {action[0]}")
    print(f"  Position: {info['position']}")
    print(f"  Reward: {reward:.4f}")
    print(f"  Terminated: {terminated}\n")

### 5.3 Targeted Actions - Only Depolarizing Noise

In [ ]:
# Reset to a new circuit
obs, info = env.reset(options={"circuit_idx": 3})

print("Applying only depolarizing noise:\n")

for i in range(3):
    # Action: [epsilon_x, epsilon_z, reset, depolarizing]
    # Set only depolarizing to non-zero
    action = np.array([[0.0, 0.0, 0.0, 0.5]])  # Only depolarizing
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Step {i+1}:")
    print(f"  Depolarizing: {action[0, 3]:.3f}")
    print(f"  Position: {info['position']}")
    print(f"  Terminated: {terminated}\n")

### 5.4 Targeted Actions - Pauli Noise Only

In [ ]:
# Reset to a new circuit
obs, info = env.reset(options={"circuit_idx": 7})

print("Applying only Pauli X and Z noise:\n")

for i in range(3):
    # Action: [epsilon_x, epsilon_z, reset, depolarizing]
    # Set only epsilon_x and epsilon_z
    action = np.array([[0.3, 0.4, 0.0, 0.0]])  # Only Pauli noise
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Step {i+1}:")
    print(f"  Epsilon X: {action[0, 0]:.3f}, Epsilon Z: {action[0, 1]:.3f}")
    print(f"  Position: {info['position']}")
    print(f"  Terminated: {terminated}\n")

### 5.5 Targeted Actions - Reset/Amplitude Damping

In [ ]:
# Reset to a new circuit
obs, info = env.reset(options={"circuit_idx": 10})

print("Applying only reset noise (amplitude damping):\n")

for i in range(3):
    # Action: [epsilon_x, epsilon_z, reset, depolarizing]
    # Set only reset parameter
    action = np.array([[0.0, 0.0, 0.6, 0.0]])  # Only reset
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Step {i+1}:")
    print(f"  Reset: {action[0, 2]:.3f}")
    print(f"  Position: {info['position']}")
    print(f"  Terminated: {terminated}\n")

### 5.6 Mixed Actions

In [ ]:
# Reset to a new circuit
obs, info = env.reset(options={"circuit_idx": 1})

print("Applying mixed noise parameters:\n")

# Define different action profiles
actions = [
    np.array([[0.2, 0.1, 0.0, 0.3]]),  # Mostly depolarizing
    np.array([[0.5, 0.5, 0.0, 0.0]]),  # Balanced Pauli
    np.array([[0.1, 0.1, 0.7, 0.1]]),  # Mostly reset
]

for i, action in enumerate(actions):
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Action {i+1}: {action[0]}")
    print(f"  Position: {info['position']}")
    print(f"  Reward: {reward:.4f}")
    print(f"  Terminated: {terminated}\n")

## 6. Complete Episode Example <a id="episode"></a>

Let's run a complete episode and track all the information.

In [ ]:
# Reset environment
obs, info = env.reset()

print(f"Starting episode with circuit {info['circuit_idx']}")
print(f"Circuit length: {info['circuit_length']} gates\n")

# Track episode data
observations = [obs]
actions = []
rewards = []
positions = [0]

# Run episode
terminated = False
step = 0

while not terminated:
    # Sample random action
    action = env.action_space.sample()
    
    # Take step
    obs, reward, terminated, truncated, info = env.step(action)
    
    # Store data
    observations.append(obs)
    actions.append(action)
    rewards.append(reward)
    positions.append(info['position'])
    
    step += 1

print(f"Episode completed!")
print(f"Total steps: {step}")
print(f"Final reward: {rewards[-1]:.4f}")
print(f"\nReward per step:")
for i, r in enumerate(rewards):
    print(f"  Step {i+1}: {r:.4f}")

### Visualize Actions and Rewards

In [ ]:
actions_array = np.array(actions)[:, 0, :]  # (steps, 4)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot actions
ax = axes[0]
steps = range(1, len(actions) + 1)
ax.plot(steps, actions_array[:, 0], 'o-', label='Epsilon X', alpha=0.7)
ax.plot(steps, actions_array[:, 1], 's-', label='Epsilon Z', alpha=0.7)
ax.plot(steps, actions_array[:, 2], '^-', label='Reset', alpha=0.7)
ax.plot(steps, actions_array[:, 3], 'd-', label='Depolarizing', alpha=0.7)
ax.set_xlabel('Step')
ax.set_ylabel('Action Value')
ax.set_title('Actions per Step')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot rewards
ax = axes[1]
ax.plot(steps, rewards, 'ko-', alpha=0.7)
ax.set_xlabel('Step')
ax.set_ylabel('Reward')
ax.set_title('Reward per Step')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Note: Reward is only given at the terminal state (last step).")

## 7. Working with Validation Circuits <a id="validation"></a>

The environment automatically splits circuits into training and validation sets.

In [ ]:
print(f"Total circuits: {env.n_circuits}")
print(f"Training circuits: {env.n_circuits_train}")
print(f"Validation circuits: {env.n_validation_circuits}")
print(f"\nValidation circuit indices:")

for i in range(env.n_validation_circuits):
    val_idx = env.get_validation_circuit(i)
    print(f"  Val circuit {i}: index {val_idx}")

### Run Episode on Validation Circuit

In [ ]:
# Get first validation circuit
val_idx = env.get_validation_circuit(0)

# Reset to validation circuit
obs, info = env.reset(options={"circuit_idx": val_idx})

print(f"Running on validation circuit {val_idx}")
print(f"Circuit length: {info['circuit_length']}\n")

# Run episode
terminated = False
step = 0

while not terminated:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    step += 1

print(f"Validation episode completed in {step} steps")
print(f"Final reward: {reward:.4f}")

## 8. Custom Reward Functions <a id="reward"></a>

Compare different reward configurations.

In [ ]:
# Test different reward functions
reward_configs = [
    RewardConfig(metric="mse", function="inverted", alpha=10.0),
    RewardConfig(metric="trace", function="inverted", alpha=20.0),
    RewardConfig(metric="fidelity", function="linear", alpha=1.0),
]

results = []

for i, reward_config in enumerate(reward_configs):
    # Create environment
    test_env = create_quantum_circuit_env(
        dataset=dataset,
        primitive_gates=["rx", "rz"],
        env_config=env_config,
        reward_config=reward_config,
    )
    
    # Run episode with fixed actions (for comparison)
    obs, info = test_env.reset(seed=42, options={"circuit_idx": 0})
    np.random.seed(42)  # Fix random actions
    
    terminated = False
    while not terminated:
        action = test_env.action_space.sample()
        obs, reward, terminated, truncated, info = test_env.step(action)
    
    results.append({
        'metric': reward_config.metric,
        'function': reward_config.function,
        'alpha': reward_config.alpha,
        'reward': reward,
    })
    
    print(f"Config {i+1}: metric={reward_config.metric}, "
          f"function={reward_config.function}, alpha={reward_config.alpha}")
    print(f"  Final reward: {reward:.4f}\n")

## 9. Multi-Qubit Environments <a id="multiqubit"></a>

Create and interact with multi-qubit environments.

In [ ]:
# Create 2-qubit dataset
dataset_config_2q = DatasetConfig(
    n_circuits=10,
    qubits=2,
    moments=8,
    clifford=True,
    seed=42
)

noise_config_2q = NoiseConfig(
    primitive_gates=["rx", "rz", "cz"],
    dep_lambda=0.03,
    seed=42
)

generator_2q = DatasetGenerator(dataset_config_2q, noise_config_2q)
dataset_2q = generator_2q.generate(verbose=True)

# Create 2-qubit environment
env_2q = create_quantum_circuit_env(
    dataset=dataset_2q,
    primitive_gates=["rx", "rz", "cz"],
    env_config=env_config,
    reward_config=reward_config,
)

print(f"\n2-Qubit Environment:")
print(f"  Observation space: {env_2q.observation_space.shape}")
print(f"  Action space: {env_2q.action_space.shape}")

### Different Actions per Qubit

In [ ]:
# Reset to a circuit
obs, info = env_2q.reset()

print(f"Applying different noise to each qubit:\n")

for i in range(3):
    # Different actions for each qubit
    action = np.array([
        [0.3, 0.0, 0.0, 0.0],  # Qubit 0: Only epsilon_x
        [0.0, 0.0, 0.0, 0.4],  # Qubit 1: Only depolarizing
    ])
    
    obs, reward, terminated, truncated, info = env_2q.step(action)
    
    print(f"Step {i+1}:")
    print(f"  Qubit 0 noise: {action[0]}")
    print(f"  Qubit 1 noise: {action[1]}")
    print(f"  Position: {info['position']}")
    print(f"  Terminated: {terminated}\n")

## 10. Advanced: Action Strategies <a id="strategies"></a>

Demonstrate different action selection strategies.

### Strategy 1: Zero Actions (No Noise)

In [ ]:
obs, info = env.reset(seed=100)
print("Strategy: Zero actions (no noise added)\n")

terminated = False
while not terminated:
    action = np.zeros((1, 4))  # No noise
    obs, reward, terminated, truncated, info = env.step(action)

print(f"Final reward with zero actions: {reward:.4f}")

### Strategy 2: Constant Actions

In [ ]:
obs, info = env.reset(seed=100)
print("Strategy: Constant moderate noise\n")

terminated = False
while not terminated:
    action = np.array([[0.2, 0.2, 0.1, 0.3]])  # Constant noise
    obs, reward, terminated, truncated, info = env.step(action)

print(f"Final reward with constant actions: {reward:.4f}")

### Strategy 3: Adaptive (Simple Rule-Based)

In [ ]:
obs, info = env.reset(seed=100)
print("Strategy: Adaptive based on gate type\n")

terminated = False
step = 0

while not terminated:
    # Simple rule: check if current gate is RX or RZ
    # (This is just an example - in practice you'd use the observation)
    
    if step % 2 == 0:
        # Even steps: more depolarizing
        action = np.array([[0.1, 0.1, 0.0, 0.5]])
    else:
        # Odd steps: more Pauli
        action = np.array([[0.4, 0.4, 0.0, 0.1]])
    
    obs, reward, terminated, truncated, info = env.step(action)
    step += 1

print(f"Final reward with adaptive actions: {reward:.4f}")

### Compare Strategies

In [ ]:
strategies = ['Zero', 'Constant', 'Adaptive']
strategy_rewards = []  # You would collect these from above

# Run all strategies on same circuit
for strategy_name in strategies:
    obs, info = env.reset(seed=100)
    
    terminated = False
    step = 0
    
    while not terminated:
        if strategy_name == 'Zero':
            action = np.zeros((1, 4))
        elif strategy_name == 'Constant':
            action = np.array([[0.2, 0.2, 0.1, 0.3]])
        else:  # Adaptive
            if step % 2 == 0:
                action = np.array([[0.1, 0.1, 0.0, 0.5]])
            else:
                action = np.array([[0.4, 0.4, 0.0, 0.1]])
        
        obs, reward, terminated, truncated, info = env.step(action)
        step += 1
    
    strategy_rewards.append(reward)

# Plot comparison
plt.figure(figsize=(10, 6))
plt.bar(strategies, strategy_rewards, color=['blue', 'green', 'orange'], alpha=0.7)
plt.xlabel('Strategy')
plt.ylabel('Final Reward')
plt.title('Comparison of Action Strategies')
plt.grid(True, alpha=0.3, axis='y')
plt.show()

for name, reward in zip(strategies, strategy_rewards):
    print(f"{name}: {reward:.4f}")

## Summary

This notebook demonstrated:
1. ✅ Creating a gymnasium environment from a quantum circuit dataset
2. ✅ Understanding observation and action spaces
3. ✅ Performing different types of actions on circuits:
   - Random actions
   - Targeted noise (depolarizing only, Pauli only, reset only)
   - Mixed noise parameters
4. ✅ Running complete episodes and tracking results
5. ✅ Working with validation circuits
6. ✅ Using different reward functions
7. ✅ Multi-qubit environments with per-qubit actions
8. ✅ Comparing different action strategies

### Next Steps
- Train an RL agent using Stable-Baselines3 or similar library
- Implement custom action selection policies
- Analyze the learned noise models
- Compare with ground truth noise parameters